# Generate Paper Embeddings with SPECTER2

This notebook creates [SPECTER2](https://github.com/allenai/SPECTER2) paper embeddings from a CSV file. Run it in Google Colab with a free or paid GPU runtime.

**Author:** [Juan Pablo Bascur](https://jpbascur.com)  
**Source:** [github.com/jpbascur/snipets/blob/main/generate_embeddings.ipynb](https://github.com/jpbascur/snipets/blob/main/generate_embeddings.ipynb)  
**License:** MIT

**Input:** a CSV file with these columns: `id`, `title`, `abstract`.

**Output:** a CSV file with no header row. The first column is the paper id, followed by 768 embedding values.

## Workflow

1. Open the notebook in Google Colab.
2. In the Colab menu, choose **Runtime > Change runtime type**.
3. Set **Hardware accelerator** to **T4 GPU** or another available GPU.
4. Click **Save**.
5. Run the code cell below.
6. A file upload prompt will appear at the bottom of the cell output — scroll down to find it. Click **Choose Files** and select your input CSV.
7. When the embeddings are finished, Colab will download `embeddings.csv` automatically.

## Notes

- The printed device line shows whether the notebook is using a GPU or CPU. If it says `Using GPU`, the GPU is active, which is usually much faster.
- The first run may take a few minutes because the model is downloaded from Hugging Face.
- If you run out of GPU memory, reduce `BATCH_SIZE` to `32`, `16`, or `8` at the top of the code cell.
- If your CSV uses different column names, rename them before running this notebook.

In [ ]:
BATCH_SIZE = 64

# Model constants: do not change these.
MODEL_NAME = 'allenai/specter2_base'
ADAPTER_NAME = 'allenai/specter2'
ADAPTER_ALIAS = 'proximity'

# Install dependencies
import subprocess
subprocess.run(['pip', 'install', '-q', 'adapters==1.3.0', 'transformers==4.57.6'], check=True)

# Imports
import io
import numpy as np
import pandas as pd
import torch
from adapters import AutoAdapterModel
from transformers import AutoTokenizer

# Device check
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('Using CPU. This will work, but it may be slow for large CSV files.')

# Load input CSV
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('No file was uploaded.')
filename = next(iter(uploaded))
print(f'Uploaded file: {filename}')
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Validate CSV shape
required_columns = ['id', 'title', 'abstract']
duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f'Duplicate column names found: {duplicate_columns}')

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

if df.empty:
    raise ValueError('The input CSV has no rows.')

print(f'Loaded {len(df)} papers.')

# Load model
print('Loading SPECTER2 model. The first run may take a few minutes...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoAdapterModel.from_pretrained(MODEL_NAME)
model.load_adapter(ADAPTER_NAME, source='hf', load_as=ADAPTER_ALIAS)
model.set_active_adapters(ADAPTER_ALIAS)
model.to(device)
model.eval()
embedding_size = model.config.hidden_size
print(f'Model ready. Embedding size: {embedding_size}')

# Encode papers
n = len(df)
embeddings = np.zeros((n, embedding_size), dtype=np.float32)

for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch = df.iloc[start:end]
    texts = (
        batch['title'].fillna('').astype(str)
        + ' [SEP] '
        + batch['abstract'].fillna('').astype(str)
    ).tolist()

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt',
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)

    embeddings[start:end] = output.last_hidden_state[:, 0, :].detach().cpu().numpy()
    print(f'Encoded {end} / {n}')

# Save and download output CSV
ids = df['id'].astype(str).reset_index(drop=True)
output_df = pd.concat([ids, pd.DataFrame(embeddings)], axis=1)
output_df.to_csv('embeddings.csv', index=False, header=False, float_format='%.8f')

print('Saved embeddings to embeddings.csv')
files.download('embeddings.csv')